# AI-Powered Network Intrusion Detection System

## Notebook 04 – Preprocessing & Model Preparation

**Author:** Jithesh  
**Role:** Member A – ML & Data Engineer  
**Dataset:** CICIDS-2017  

---

### Purpose

The purpose of this notebook is to prepare the feature-engineered CICIDS-2017 network traffic dataset for machine learning model training.

The notebook performs the required preprocessing after feature engineering, including reproducible dataset splitting, numerical scaling, class-imbalance handling, and creation of reusable preprocessing artifacts. The resulting outputs will provide a consistent and validated foundation for subsequent model training and evaluation.

---

### Notebook Objective

This notebook continues from Notebook 03 and prepares the persisted feature-engineered dataset for machine learning model training.

The main objectives are to:

1. Load and verify the persisted feature-engineered dataset.
2. Separate input features and target labels.
3. Create reproducible training, validation, and test splits.
4. Apply numerical scaling using parameters learned only from the training data.
5. Address the severe class imbalance using an appropriate training-only strategy.
6. Create reusable preprocessing artifacts for later model training and inference.
7. Validate and persist the resulting model-ready data and preprocessing artifacts.

The feature-engineered dataset generated by Notebook 03 is preserved as the canonical input and is not modified by this notebook.

## 2. Environment Setup

This section initializes the Colab environment, imports the libraries required for preprocessing and model preparation, and defines the project storage paths.

The notebook uses the persisted feature-engineered dataset generated by Notebook 03 as its input.

All project artifacts continue to use the established Google Drive directory structure to ensure reproducibility across Colab runtime sessions.

In [1]:
# ============================================================
# 2.1 Environment Setup
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

# Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Project paths
PROJECT_PATH = "/content/drive/MyDrive/AI-Network-Intrusion-Detection-System"

DATA_PATH = os.path.join(PROJECT_PATH, "data")
RAW_DATA_PATH = os.path.join(DATA_PATH, "raw")
PROCESSED_DATA_PATH = os.path.join(DATA_PATH, "processed")

OUTPUTS_PATH = os.path.join(PROJECT_PATH, "outputs")
MODELS_PATH = os.path.join(PROJECT_PATH, "trained_models")

# Create directories if they do not already exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(OUTPUTS_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

print("=" * 70)
print("ENVIRONMENT SETUP")
print("=" * 70)

print("Project path:")
print(PROJECT_PATH)

print("\nProcessed data path:")
print(PROCESSED_DATA_PATH)

print("\nOutputs path:")
print(OUTPUTS_PATH)

print("\nModels path:")
print(MODELS_PATH)

print("\nEnvironment setup completed successfully.")
print("=" * 70)

Mounted at /content/drive
ENVIRONMENT SETUP
Project path:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System

Processed data path:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/data/processed

Outputs path:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs

Models path:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/trained_models

Environment setup completed successfully.


### 2.2 Define Stage 1 Input Artifact

The primary input for this notebook is the feature-engineered dataset persisted by Notebook 03.

The dataset contains the final 63 input features and the `Target` variable.

The original Stage 1 artifact is treated as read-only. Subsequent preprocessing operations will create new representations rather than modifying the persisted Stage 1 dataset.

In [2]:
# ============================================================
# 2.2 Define Stage 1 Input Artifact
# ============================================================

STAGE1_DATA_PATH = os.path.join(
    PROCESSED_DATA_PATH,
    "CICIDS2017_stage1_feature_engineered.parquet"
)

STAGE1_METADATA_PATH = os.path.join(
    PROCESSED_DATA_PATH,
    "CICIDS2017_stage1_metadata.json"
)

TARGET_COLUMN = "Target"

print("=" * 70)
print("STAGE 1 INPUT ARTIFACT")
print("=" * 70)

print("Feature-engineered dataset:")
print(STAGE1_DATA_PATH)

print("\nMetadata:")
print(STAGE1_METADATA_PATH)

print("\nDataset exists:", os.path.exists(STAGE1_DATA_PATH))
print("Metadata exists:", os.path.exists(STAGE1_METADATA_PATH))

if not os.path.exists(STAGE1_DATA_PATH):
    raise FileNotFoundError(
        "Stage 1 feature-engineered dataset was not found."
    )

if not os.path.exists(STAGE1_METADATA_PATH):
    raise FileNotFoundError(
        "Stage 1 metadata file was not found."
    )

print("\nInput artifact verification: PASSED")
print("=" * 70)

STAGE 1 INPUT ARTIFACT
Feature-engineered dataset:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/data/processed/CICIDS2017_stage1_feature_engineered.parquet

Metadata:
/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/data/processed/CICIDS2017_stage1_metadata.json

Dataset exists: True
Metadata exists: True

Input artifact verification: PASSED


### 2.3 Load and Verify Stage 1 Metadata

**Purpose:**  
Verify that the persisted Stage 1 feature-engineered dataset matches the metadata produced by Notebook 03 before any further preprocessing is performed.

The metadata provides the expected dataset size, target column, input feature count, removed features, transformations, and validation status. This verification helps ensure that Notebook 04 is operating on the correct and reproducible Stage 1 artifact.

In [3]:
# ============================================================
# 2.3 Load and Verify Stage 1 Metadata
# ============================================================

with open(STAGE1_METADATA_PATH, "r") as f:
    stage1_metadata = json.load(f)

print("=" * 70)
print("STAGE 1 METADATA VERIFICATION")
print("=" * 70)

print("\nDataset:", stage1_metadata["dataset"]["name"])
print("Records:", stage1_metadata["dataset"]["number_of_records"])
print("Input features:", stage1_metadata["dataset"]["number_of_input_features"])
print("Target column:", stage1_metadata["dataset"]["target_column"])

print("\nRemoved features:")
for feature in stage1_metadata["removed_features"]:
    print(" -", feature)

print("\nFeature transformation:")
print(stage1_metadata["feature_transformation"])

print("\nValidation:")
for key, value in stage1_metadata["validation"].items():
    print(f" - {key}: {value}")

print("\nNumber of feature columns listed in metadata:",
      len(stage1_metadata["feature_columns"]))

print("=" * 70)

STAGE 1 METADATA VERIFICATION

Dataset: CICIDS2017
Records: 2830743
Input features: 63
Target column: Target

Removed features:
 - CWE Flag Count
 - SYN Flag Count

Feature transformation:
log1p applied to selected highly skewed non-binary non-negative features

Validation:
 - nan_values: 0
 - infinite_values: 0
 - negative_values: 0
 - non_numeric_features: 0

Number of feature columns listed in metadata: 63


In [4]:
# ============================================================
# 2.4 Validate Stage 1 Metadata Consistency
# ============================================================

assert stage1_metadata["dataset"]["name"] == "CICIDS2017"
assert stage1_metadata["dataset"]["target_column"] == TARGET_COLUMN
assert stage1_metadata["dataset"]["number_of_input_features"] == 63
assert len(stage1_metadata["feature_columns"]) == 63

assert stage1_metadata["validation"]["nan_values"] == 0
assert stage1_metadata["validation"]["infinite_values"] == 0
assert stage1_metadata["validation"]["negative_values"] == 0
assert stage1_metadata["validation"]["non_numeric_features"] == 0

print("Stage 1 metadata consistency check: PASSED")

Stage 1 metadata consistency check: PASSED


### 2.5 Load Stage 1 Feature-Engineered Dataset

**Purpose:**  
Load the persisted feature-engineered dataset produced by Notebook 03 for model-preparation preprocessing.

The dataset is loaded only after verifying its existence and metadata consistency. The loaded data will be checked against the expected schema before applying any additional preprocessing.

In [5]:
# ============================================================
# 2.5 Load Stage 1 Feature-Engineered Dataset
# ============================================================

print("=" * 70)
print("LOADING STAGE 1 FEATURE-ENGINEERED DATASET")
print("=" * 70)

df = pd.read_parquet(STAGE1_DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

print("\nMemory usage:")
print(f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

print("=" * 70)

LOADING STAGE 1 FEATURE-ENGINEERED DATASET

Dataset loaded successfully.
Shape: (2830743, 64)

Memory usage:
1.35 GB


### 2.6 Verify Stage 1 Dataset Structure

**Purpose:**  
Confirm that the loaded feature-engineered dataset has the expected number of records, feature columns, target column, and numeric data types before model-preparation preprocessing begins.

In [6]:
# ============================================================
# 2.6 Verify Stage 1 Dataset Structure
# ============================================================

expected_features = stage1_metadata["feature_columns"]

print("=" * 70)
print("STAGE 1 DATASET STRUCTURE VERIFICATION")
print("=" * 70)

print("\nActual shape:", df.shape)

# Record count
assert len(df) == stage1_metadata["dataset"]["number_of_records"]

# Target column
assert TARGET_COLUMN in df.columns

# Feature columns
actual_features = [col for col in df.columns if col != TARGET_COLUMN]

assert actual_features == expected_features

# Total column count
assert len(df.columns) == len(expected_features) + 1

# Numeric validation
non_numeric_columns = df.select_dtypes(exclude="number").columns.tolist()

assert len(non_numeric_columns) == 0

print("\nRecord count check: PASSED")
print("Target column check: PASSED")
print("Feature column check: PASSED")
print("Column count check: PASSED")
print("Numeric feature check: PASSED")

print("\nFinal verified shape:", df.shape)

print("=" * 70)

STAGE 1 DATASET STRUCTURE VERIFICATION

Actual shape: (2830743, 64)

Record count check: PASSED
Target column check: PASSED
Feature column check: PASSED
Column count check: PASSED
Numeric feature check: PASSED

Final verified shape: (2830743, 64)


### 2.7 Inspect Target Distribution

**Purpose:**  
Examine the distribution of the target classes in the feature-engineered dataset.

This step establishes the degree of class imbalance before model-preparation decisions are made. The original class distribution is preserved for reference and will not be modified at this stage.

In [7]:
# ============================================================
# 2.7 Inspect Target Distribution
# ============================================================

target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_percentages = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages.round(4)
})

print("=" * 70)
print("TARGET CLASS DISTRIBUTION")
print("=" * 70)

print(target_distribution)

print("\nNumber of target classes:", df[TARGET_COLUMN].nunique())
print("Total samples:", target_counts.sum())

print("=" * 70)

TARGET CLASS DISTRIBUTION
          count  percentage
Target                     
0       2273097     80.3004
1          1966      0.0695
2        128027      4.5227
3         10293      0.3636
4        231073      8.1630
5          5499      0.1943
6          5796      0.2048
7          7938      0.2804
8            11      0.0004
9            36      0.0013
10       158930      5.6144
11         5897      0.2083
12         1507      0.0532
13           21      0.0007
14          652      0.0230

Number of target classes: 15
Total samples: 2830743


### 2.8 Class Imbalance Strategy

**Purpose:**  
Define the class-imbalance handling strategy for model preparation based on the observed target distribution.

The CICIDS2017 target contains 15 classes with substantial imbalance, with the majority class representing approximately 80% of the dataset and several minority classes containing very few samples.

Because the dataset is large and highly imbalanced, naive oversampling of all minority classes could substantially increase memory usage and introduce excessive synthetic samples.

Therefore, class balancing will not be performed on the complete dataset at this stage. The original class distribution will be preserved, and imbalance handling will be incorporated during model training and evaluation using appropriate techniques such as class-weighted learning and stratified data splitting where supported.

In [8]:
# ============================================================
# 2.8 Class Imbalance Strategy Verification
# ============================================================

majority_class = target_counts.idxmax()
majority_count = target_counts.max()

minority_class = target_counts.idxmin()
minority_count = target_counts.min()

imbalance_ratio = majority_count / minority_count

print("=" * 70)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 70)

print(f"\nNumber of classes: {len(target_counts)}")

print(
    f"Majority class: {majority_class} "
    f"({majority_count:,} samples)"
)

print(
    f"Minority class: {minority_class} "
    f"({minority_count:,} samples)"
)

print(f"Majority/minority ratio: {imbalance_ratio:,.2f}")

print("\nClass balancing decision:")
print("- Preserve the original dataset distribution.")
print("- Do not oversample the complete dataset.")
print("- Use stratified splitting where applicable.")
print("- Use class-weighted learning where supported.")

print("\nClass imbalance strategy: DOCUMENTED")
print("=" * 70)

CLASS IMBALANCE ANALYSIS

Number of classes: 15
Majority class: 0 (2,273,097 samples)
Minority class: 8 (11 samples)
Majority/minority ratio: 206,645.18

Class balancing decision:
- Preserve the original dataset distribution.
- Do not oversample the complete dataset.
- Use stratified splitting where applicable.
- Use class-weighted learning where supported.

Class imbalance strategy: DOCUMENTED


## 2.9 Train-Test Split

**Purpose:**

Split the verified feature-engineered dataset into training and testing subsets while preserving the original class distribution.

A stratified split is used because the CICIDS2017 target variable is highly imbalanced. Stratification ensures that the relative distribution of the 15 target classes is maintained across the training and testing sets as far as the available samples permit.

The split is performed before any model-specific preprocessing such as scaling or normalization to prevent data leakage. The test set will remain isolated and will only be used for final model evaluation.

In [9]:
# ============================================================
# 2.9 Train-Test Split
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

# Separate input features and target
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("\nOriginal dataset shape:")
print("X:", X.shape)
print("y:", y.shape)

# Stratified 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nSplit completed successfully.")

print("\nTraining set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\n" + "=" * 70)

TRAIN-TEST SPLIT

Original dataset shape:
X: (2830743, 63)
y: (2830743,)

Split completed successfully.

Training set:
X_train: (2264594, 63)
y_train: (2264594,)

Testing set:
X_test: (566149, 63)
y_test: (566149,)



In [10]:
# ============================================================
# Validate Train-Test Split
# ============================================================

print("=" * 70)
print("TRAIN-TEST SPLIT VALIDATION")
print("=" * 70)

# Verify that the number of samples is preserved
assert len(X_train) + len(X_test) == len(df)
assert len(y_train) + len(y_test) == len(df)

# Verify feature and target dimensions
assert X_train.shape[1] == X_test.shape[1]
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Verify that all target classes are present in both subsets
assert set(y_train.unique()) == set(y.unique())
assert set(y_test.unique()) == set(y.unique())

print("\nSample count check: PASSED")
print("Feature dimension check: PASSED")
print("Training target alignment: PASSED")
print("Testing target alignment: PASSED")
print("Class presence check: PASSED")

print("\nTrain-test split validation completed successfully.")
print("=" * 70)

TRAIN-TEST SPLIT VALIDATION

Sample count check: PASSED
Feature dimension check: PASSED
Training target alignment: PASSED
Testing target alignment: PASSED
Class presence check: PASSED

Train-test split validation completed successfully.


In [11]:
# ============================================================
# Compare Class Distribution After Stratified Split
# ============================================================

train_distribution = y_train.value_counts(normalize=True).sort_index() * 100
test_distribution = y_test.value_counts(normalize=True).sort_index() * 100

split_distribution = pd.DataFrame({
    "Original (%)": y.value_counts(normalize=True).sort_index() * 100,
    "Train (%)": train_distribution,
    "Test (%)": test_distribution
}).round(4)

print("=" * 70)
print("CLASS DISTRIBUTION AFTER STRATIFIED SPLIT")
print("=" * 70)

print(split_distribution)

print("\nStratified split completed.")
print("Original class distribution is preserved approximately in both subsets.")
print("=" * 70)

CLASS DISTRIBUTION AFTER STRATIFIED SPLIT
        Original (%)  Train (%)  Test (%)
Target                                   
0            80.3004    80.3004   80.3004
1             0.0695     0.0695    0.0694
2             4.5227     4.5227    4.5228
3             0.3636     0.3636    0.3637
4             8.1630     8.1630    8.1630
5             0.1943     0.1943    0.1943
6             0.2048     0.2048    0.2047
7             0.2804     0.2804    0.2805
8             0.0004     0.0004    0.0004
9             0.0013     0.0013    0.0012
10            5.6144     5.6144    5.6144
11            0.2083     0.2083    0.2082
12            0.0532     0.0533    0.0532
13            0.0007     0.0008    0.0007
14            0.0230     0.0231    0.0230

Stratified split completed.
Original class distribution is preserved approximately in both subsets.


## 2.10 Feature Scaling

**Purpose:**

Standardize the numerical input features using statistics learned only from the training set.

The scaler is fitted exclusively on `X_train` and then applied to both `X_train` and `X_test`. This prevents information from the test set from influencing the preprocessing step and avoids data leakage.

Standardized features are useful for scale-sensitive machine learning algorithms such as Logistic Regression, SVM, and KNN.

The fitted scaler will be persisted as a preprocessing artifact so that the same transformation can be applied during inference.

In [12]:
# ============================================================
# 2.10 Feature Scaling
# ============================================================

from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("FEATURE SCALING")
print("=" * 70)

print("\nOriginal training shape:", X_train.shape)
print("Original testing shape:", X_test.shape)

# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data using the same fitted scaler
X_test_scaled = scaler.transform(X_test)

print("\nScaling completed successfully.")

print("\nScaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nScaler fitted only on training data.")
print("Test data transformed using training-set statistics.")

print("=" * 70)

FEATURE SCALING

Original training shape: (2264594, 63)
Original testing shape: (566149, 63)

Scaling completed successfully.

Scaled training shape: (2264594, 63)
Scaled testing shape: (566149, 63)

Scaler fitted only on training data.
Test data transformed using training-set statistics.


### 2.10.1 Validate Feature Scaling

Verify that scaling was applied correctly and that the transformation preserved the feature dimensions.

The training features should have approximately zero mean and unit standard deviation after standardization.

The testing data should use the same scaler fitted on the training data.

In [13]:
# ============================================================
# 2.10.1 Validate Feature Scaling
# ============================================================

print("=" * 70)
print("FEATURE SCALING VALIDATION")
print("=" * 70)

# Shape validation
assert X_train_scaled.shape == X_train.shape
assert X_test_scaled.shape == X_test.shape

# Check that scaled training data has approximately zero mean
train_means = X_train_scaled.mean(axis=0)

# Check that scaled training data has approximately unit standard deviation
train_stds = X_train_scaled.std(axis=0)

# Numerical tolerance for floating-point calculations
assert np.allclose(train_means, 0, atol=1e-6)
assert np.allclose(train_stds, 1, atol=1e-6)

print("\nShape check: PASSED")
print("Training mean check: PASSED")
print("Training standard deviation check: PASSED")

print("\nMaximum absolute training mean:",
      np.max(np.abs(train_means)))

print("Maximum deviation from unit standard deviation:",
      np.max(np.abs(train_stds - 1)))

print("\nFeature scaling validation completed successfully.")

print("=" * 70)

FEATURE SCALING VALIDATION

Shape check: PASSED
Training mean check: PASSED
Training standard deviation check: PASSED

Maximum absolute training mean: 1.379456204156965e-15
Maximum deviation from unit standard deviation: 2.9976021664879227e-15

Feature scaling validation completed successfully.


## 2.11 Persist Model-Preparation Artifacts

**Purpose:**

Persist the finalized model-ready training and testing datasets, target arrays, fitted feature scaler, and feature metadata.

These artifacts will serve as the standardized input for subsequent model-training notebooks. Saving the preprocessing outputs ensures that later experiments use the same train-test split and feature transformation without repeating preprocessing.

The original Stage 1 feature-engineered dataset remains unchanged.

In [14]:
# ============================================================
# 2.11 Persist Model-Preparation Artifacts
# ============================================================

import os
import json
import joblib
import numpy as np

# Create dedicated directory for model-preparation artifacts
MODEL_PREP_PATH = os.path.join(
    OUTPUTS_PATH,
    "model_preparation"
)

os.makedirs(MODEL_PREP_PATH, exist_ok=True)

print("=" * 70)
print("PERSISTING MODEL-PREPARATION ARTIFACTS")
print("=" * 70)

# ------------------------------------------------------------
# Convert scaled features to float32 to reduce storage size
# while preserving sufficient numerical precision for ML.
# ------------------------------------------------------------

X_train_final = X_train_scaled.astype(np.float32)
X_test_final = X_test_scaled.astype(np.float32)

# Ensure targets have a compact integer representation
y_train_final = y_train.to_numpy(dtype=np.int8)
y_test_final = y_test.to_numpy(dtype=np.int8)

# ------------------------------------------------------------
# Save feature matrices
# ------------------------------------------------------------

X_train_path = os.path.join(
    MODEL_PREP_PATH,
    "X_train_scaled.npy"
)

X_test_path = os.path.join(
    MODEL_PREP_PATH,
    "X_test_scaled.npy"
)

y_train_path = os.path.join(
    MODEL_PREP_PATH,
    "y_train.npy"
)

y_test_path = os.path.join(
    MODEL_PREP_PATH,
    "y_test.npy"
)

np.save(X_train_path, X_train_final)
np.save(X_test_path, X_test_final)
np.save(y_train_path, y_train_final)
np.save(y_test_path, y_test_final)

# ------------------------------------------------------------
# Save fitted scaler
# ------------------------------------------------------------

SCALER_PATH = os.path.join(
    MODEL_PREP_PATH,
    "standard_scaler.joblib"
)

joblib.dump(scaler, SCALER_PATH)

# ------------------------------------------------------------
# Save feature metadata
# ------------------------------------------------------------

FEATURE_METADATA_PATH = os.path.join(
    MODEL_PREP_PATH,
    "feature_metadata.json"
)

feature_metadata = {
    "dataset": "CICIDS2017",
    "number_of_features": len(expected_features),
    "feature_columns": expected_features,
    "target_column": TARGET_COLUMN,
    "scaling": "StandardScaler",
    "scaler_fitted_on": "training_data_only",
    "train_samples": int(len(y_train_final)),
    "test_samples": int(len(y_test_final)),
    "feature_dtype": "float32",
    "target_dtype": "int8"
}

with open(FEATURE_METADATA_PATH, "w") as f:
    json.dump(feature_metadata, f, indent=2)

print("\nArtifacts saved successfully.")

print("\nSaved files:")
print("-", X_train_path)
print("-", X_test_path)
print("-", y_train_path)
print("-", y_test_path)
print("-", SCALER_PATH)
print("-", FEATURE_METADATA_PATH)

print("\nTraining features:", X_train_final.shape)
print("Testing features:", X_test_final.shape)
print("Training targets:", y_train_final.shape)
print("Testing targets:", y_test_final.shape)

print("\n" + "=" * 70)

PERSISTING MODEL-PREPARATION ARTIFACTS

Artifacts saved successfully.

Saved files:
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/X_train_scaled.npy
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/X_test_scaled.npy
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/y_train.npy
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/y_test.npy
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/standard_scaler.joblib
- /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/outputs/model_preparation/feature_metadata.json

Training features: (2264594, 63)
Testing features: (566149, 63)
Training targets: (2264594,)
Testing targets: (566149,)



### 2.11.1 Validate Persisted Artifacts

**Purpose:**

Verify that all model-preparation artifacts were successfully persisted and can be reloaded without loss of structure or consistency.

The validation confirms that the saved training/testing features, target arrays, scaler, and feature metadata are suitable for reuse by subsequent model-training notebooks.

In [15]:
# ============================================================
# 2.11.1 Validate Persisted Artifacts
# ============================================================

print("=" * 70)
print("MODEL-PREPARATION ARTIFACT VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

required_artifacts = [
    X_train_path,
    X_test_path,
    y_train_path,
    y_test_path,
    SCALER_PATH,
    FEATURE_METADATA_PATH
]

for artifact in required_artifacts:
    assert os.path.exists(artifact), f"Missing artifact: {artifact}"

print("\nFile existence check: PASSED")

# ------------------------------------------------------------
# Reload saved artifacts
# ------------------------------------------------------------

X_train_check = np.load(X_train_path)
X_test_check = np.load(X_test_path)

y_train_check = np.load(y_train_path)
y_test_check = np.load(y_test_path)

scaler_check = joblib.load(SCALER_PATH)

with open(FEATURE_METADATA_PATH, "r") as f:
    metadata_check = json.load(f)

print("Artifact reload check: PASSED")

# ------------------------------------------------------------
# Validate shapes
# ------------------------------------------------------------

assert X_train_check.shape == (2264594, 63)
assert X_test_check.shape == (566149, 63)

assert y_train_check.shape == (2264594,)
assert y_test_check.shape == (566149,)

print("Feature shape check: PASSED")
print("Target shape check: PASSED")

# ------------------------------------------------------------
# Validate target consistency
# ------------------------------------------------------------

assert np.array_equal(y_train_check, y_train_final)
assert np.array_equal(y_test_check, y_test_final)

print("Target consistency check: PASSED")

# ------------------------------------------------------------
# Validate metadata
# ------------------------------------------------------------

assert metadata_check["dataset"] == "CICIDS2017"
assert metadata_check["number_of_features"] == 63
assert metadata_check["target_column"] == TARGET_COLUMN
assert metadata_check["scaling"] == "StandardScaler"
assert metadata_check["scaler_fitted_on"] == "training_data_only"

print("Metadata consistency check: PASSED")

# ------------------------------------------------------------
# Validate scaler
# ------------------------------------------------------------

assert hasattr(scaler_check, "mean_")
assert hasattr(scaler_check, "scale_")
assert len(scaler_check.mean_) == 63
assert len(scaler_check.scale_) == 63

print("Scaler validation check: PASSED")

print("\nModel-preparation artifact validation completed successfully.")

print("=" * 70)

MODEL-PREPARATION ARTIFACT VALIDATION

File existence check: PASSED
Artifact reload check: PASSED
Feature shape check: PASSED
Target shape check: PASSED
Target consistency check: PASSED
Metadata consistency check: PASSED
Scaler validation check: PASSED

Model-preparation artifact validation completed successfully.


## 2.12 Model Preparation Summary

**Purpose:**

Document the final model-ready representation produced by this notebook and provide a clear handoff to the subsequent model-training stage.

The preprocessing pipeline completed the following steps:

- Loaded the validated Stage 1 feature-engineered dataset.
- Verified dataset structure and metadata consistency.
- Analyzed the target class distribution.
- Preserved class proportions using stratified train-test splitting.
- Created training and testing feature sets with 63 input features.
- Applied `StandardScaler`, fitted exclusively on the training data.
- Applied the fitted scaler to the testing data.
- Validated the scaled feature matrices.
- Persisted the training/testing features, targets, scaler, and feature metadata.
- Reloaded and validated all persisted artifacts.

The resulting artifacts are ready for consumption by subsequent model-training notebooks.

In [16]:
# ============================================================
# 2.12 Model Preparation Summary
# ============================================================

print("=" * 70)
print("MODEL PREPARATION COMPLETE")
print("=" * 70)

print("\nDataset: CICIDS2017")

print("\nFinal model-ready data:")
print(f"- Training features: {X_train_final.shape}")
print(f"- Testing features:  {X_test_final.shape}")
print(f"- Training targets:  {y_train_final.shape}")
print(f"- Testing targets:   {y_test_final.shape}")

print("\nPreprocessing:")
print("- Stratified train-test split")
print("- StandardScaler")
print("- Scaler fitted on training data only")
print("- Test data transformed using training-set statistics")

print("\nPersisted artifacts:")
print("- X_train_scaled.npy")
print("- X_test_scaled.npy")
print("- y_train.npy")
print("- y_test.npy")
print("- standard_scaler.joblib")
print("- feature_metadata.json")

print("\nModel-preparation pipeline completed successfully.")
print("Artifacts are ready for downstream model training.")

print("=" * 70)

MODEL PREPARATION COMPLETE

Dataset: CICIDS2017

Final model-ready data:
- Training features: (2264594, 63)
- Testing features:  (566149, 63)
- Training targets:  (2264594,)
- Testing targets:   (566149,)

Preprocessing:
- Stratified train-test split
- StandardScaler
- Scaler fitted on training data only
- Test data transformed using training-set statistics

Persisted artifacts:
- X_train_scaled.npy
- X_test_scaled.npy
- y_train.npy
- y_test.npy
- standard_scaler.joblib
- feature_metadata.json

Model-preparation pipeline completed successfully.
Artifacts are ready for downstream model training.


In [17]:
# ============================================================
# 2.12 Verify and Persist Project Artifacts
# ============================================================

import os

MODEL_PREP_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "model_preparation"
)

print("=" * 70)
print("MODEL-PREPARATION ARTIFACT PERSISTENCE CHECK")
print("=" * 70)

required_artifacts = [
    "X_train_scaled.npy",
    "X_test_scaled.npy",
    "y_train.npy",
    "y_test.npy",
    "standard_scaler.joblib",
    "feature_metadata.json"
]

all_present = True

for filename in required_artifacts:
    filepath = os.path.join(MODEL_PREP_PATH, filename)

    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 ** 2)
        print(f"✓ {filename:<35} {size_mb:.2f} MB")
    else:
        print(f"✗ {filename:<35} MISSING")
        all_present = False

print("=" * 70)

if all_present:
    print("All model-preparation artifacts are present.")
    print("Artifacts are persisted and ready for reuse.")
else:
    raise FileNotFoundError(
        "One or more required model-preparation artifacts are missing."
    )

print("Artifact persistence check: PASSED")
print("=" * 70)

MODEL-PREPARATION ARTIFACT PERSISTENCE CHECK
✓ X_train_scaled.npy                  544.24 MB
✓ X_test_scaled.npy                   136.06 MB
✓ y_train.npy                         2.16 MB
✓ y_test.npy                          0.54 MB
✓ standard_scaler.joblib              0.00 MB
✓ feature_metadata.json               0.00 MB
All model-preparation artifacts are present.
Artifacts are persisted and ready for reuse.
Artifact persistence check: PASSED
